# HW #4: Trace one sentence through training and generation

202315163 김소희

`i want chinese food`를 토큰 ID로 바꾸고, 학습 입력과 정답을 만든다.
`trace_model(model, sentence)`로 입력부터 예측까지 각 단계의 텐서 크기를 확인하고,
`food` 다음에 올 토큰 중 점수가 높은 5개를 출력한다.

## 1. 데이터와 단어 ID

음식과 식당에 관한 문장 47개로 모델을 학습한다.
각 문장을 소문자로 바꾸고 공백을 기준으로 단어를 나눈다.
`<pad>`, `<bos>`, `<eos>`, `<unk>` 뒤에 나머지 단어를 사전순으로 넣고,
그 순서대로 ID를 부여한다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
if device.type == "cpu":
    # 작은 실습 모델에서 CPU 스레드 관리 비용을 줄인다.
    torch.set_num_threads(1)

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.14.0+cpu
Device: cpu


In [2]:
sentences = [
    "i want chinese food",
    "i want korean food",
    "i want italian food",
    "i want japanese food",
    "i want thai food",
    "i want indian food",

    "i want a chinese restaurant",
    "i want a korean restaurant",
    "i want an italian restaurant",
    "i want a japanese restaurant",

    "i want to eat chinese food",
    "i want to eat korean food",
    "i want to eat italian food",
    "i want to eat japanese food",

    "i would like chinese food",
    "i would like korean food",
    "i would like italian food",
    "i would like japanese food",

    "find a chinese restaurant",
    "find a korean restaurant",
    "find an italian restaurant",
    "find a japanese restaurant",

    "show me a chinese restaurant",
    "show me a korean restaurant",
    "show me an italian restaurant",
    "show me a japanese restaurant",

    "can you find a chinese restaurant",
    "can you find a korean restaurant",
    "can you find an italian restaurant",
    "can you find a japanese restaurant",

    "where can i eat chinese food",
    "where can i eat korean food",
    "where can i eat italian food",
    "where can i eat japanese food",
    "i want cheap chinese food",
    "i want cheap korean food",
    "i want expensive italian food",

    "find a cheap restaurant",
    "find a good restaurant",
    "find a nearby restaurant",

    "i want a restaurant downtown",
    "i want a restaurant nearby",
    "show me a restaurant downtown",

    "the chinese restaurant is good",
    "the korean restaurant is good",
    "the italian restaurant is expensive",
    "the thai restaurant is cheap",
]

In [3]:
SPECIAL_TOKENS = [
    "<pad>",
    "<bos>",
    "<eos>",
    "<unk>"
]

words = []

for sentence in sentences:
    words.extend(
        sentence.lower().split()
    )

vocab = SPECIAL_TOKENS + sorted(
    set(words)
)


word_to_id = {
    word: i
    for i, word in enumerate(vocab)
}

id_to_word = {
    i: word
    for word, i in word_to_id.items()
}

PAD_ID = word_to_id["<pad>"]
BOS_ID = word_to_id["<bos>"]
EOS_ID = word_to_id["<eos>"]
UNK_ID = word_to_id["<unk>"]

vocab_size = len(vocab)

print("Vocabulary size:", vocab_size)
print(word_to_id)

Vocabulary size: 33
{'<pad>': 0, '<bos>': 1, '<eos>': 2, '<unk>': 3, 'a': 4, 'an': 5, 'can': 6, 'cheap': 7, 'chinese': 8, 'downtown': 9, 'eat': 10, 'expensive': 11, 'find': 12, 'food': 13, 'good': 14, 'i': 15, 'indian': 16, 'is': 17, 'italian': 18, 'japanese': 19, 'korean': 20, 'like': 21, 'me': 22, 'nearby': 23, 'restaurant': 24, 'show': 25, 'thai': 26, 'the': 27, 'to': 28, 'want': 29, 'where': 30, 'would': 31, 'you': 32}


## 2. 학습 입력 x와 정답 y

문장 앞에 `<bos>`, 뒤에 `<eos>`를 붙인다.
`x = ids[:-1]`은 마지막 토큰을 뺀 입력이고,
`y = ids[1:]`은 첫 토큰을 뺀 정답이다.
따라서 x의 각 위치에서 y의 같은 위치에 있는 다음 토큰을 예측하도록 학습한다.

| 위치 | 입력 x | 정답 y |
|---:|---|---|
| 0 | `<bos>` | `i` |
| 1 | `i` | `want` |
| 2 | `want` | `chinese` |
| 3 | `chinese` | `food` |
| 4 | `food` | `<eos>` |

x와 y에 각각 `<pad>` 7개를 붙여 길이를 12로 맞춘다.
`CrossEntropyLoss(ignore_index=PAD_ID)`를 사용해 정답이 `<pad>`인 위치는 손실 계산에서 제외한다.

In [4]:
MAX_LEN = 12

class LanguageModelDataset(Dataset):
    def __init__(self, sentences):
        self.examples = []

        for sentence in sentences:
            tokens = sentence.lower().split()
            ids = [BOS_ID]

            for word in tokens:
                ids.append(
                    word_to_id.get(
                        word,
                        UNK_ID
                    )
                )

            ids.append(EOS_ID)

            # Input
            x = ids[:-1]

            # Next-word targets
            y = ids[1:]

            x = x[:MAX_LEN]
            y = y[:MAX_LEN]

            padding = MAX_LEN - len(x)

            x = x + [PAD_ID] * padding
            y = y + [PAD_ID] * padding

            self.examples.append(
                (
                    torch.tensor(
                        x,
                        dtype=torch.long
                    ),
                    torch.tensor(
                        y,
                        dtype=torch.long
                    )
                )
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]

In [5]:
dataset = LanguageModelDataset(sentences)
loader = DataLoader(dataset, batch_size=16, shuffle=True)
print("Training sentences:", len(dataset))

Training sentences: 47


## 3. RNN 모델 학습

`RNNLanguageModel`은 단어 ID를 64차원 임베딩으로 바꾼 뒤 RNN에 넣는다.
RNN의 hidden state는 128차원이며, 마지막 선형 계층은 각 위치에서 어휘 33개의 점수를 출력한다.
이 점수가 로짓이다.

AdamW를 사용하고 학습률은 0.003으로 설정한다. 전체 학습 데이터를 200회 반복해서 학습한다.

In [6]:
class RNNLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=64,
        hidden_dim=128
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.output = nn.Linear(
            hidden_dim,
            vocab_size
        )

    def forward(self, x):
        # x:
        # [batch, sequence]

        embedded = self.embedding(x)

        # embedded:
        # [batch, sequence, embedding_dim]

        hidden_states, hidden = self.rnn(
            embedded
        )

        # hidden_states:
        # [batch, sequence, hidden_dim]

        logits = self.output(
            hidden_states
        )

        # logits:
        # [batch, sequence, vocab_size]

        return logits

In [7]:
def train_model(
    model,
    loader,
    epochs=200,
    learning_rate=0.003
):
    model = model.to(device)

    criterion = nn.CrossEntropyLoss(
        ignore_index=PAD_ID
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate
    )

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits = model(x)

            loss = criterion(
                logits.reshape(
                    -1,
                    vocab_size
                ),
                y.reshape(-1)
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 20 == 0:
            average_loss=(total_loss/len(loader))

            print(f"Epoch: {epoch+1:3d}, Loss: "
                  f"{average_loss:.4f}")

In [8]:
rnn_model = RNNLanguageModel(vocab_size).to(device)
train_model(rnn_model, loader)

Epoch:  20, Loss: 0.6952
Epoch:  40, Loss: 0.6670
Epoch:  60, Loss: 0.6598
Epoch:  80, Loss: 0.6603
Epoch: 100, Loss: 0.6585
Epoch: 120, Loss: 0.6568
Epoch: 140, Loss: 0.6534
Epoch: 160, Loss: 0.6548
Epoch: 180, Loss: 0.6536
Epoch: 200, Loss: 0.6542


## 4. trace_model(model, sentence)

먼저 원문과 토큰 ID, 학습 입력 x, 정답 y를 출력한다.
패딩된 x를 모델에 넣어 입력, 임베딩, 로짓의 크기를 확인한다.

다음 토큰을 예측할 때는 `<bos> i want chinese food`를 입력한다.
`unsqueeze(0)`으로 배치 차원을 추가하고,
`logits[0, -1]`로 `food` 위치의 출력을 선택한다.
여기에서 점수가 가장 높은 토큰 5개를 찾고, 전체 어휘에 softmax를 적용한 확률을 함께 출력한다.
`<eos>`도 문장을 끝내는 토큰이므로 후보에 포함한다.

예측 중에는 `model.eval()`과 `torch.no_grad()`를 사용한다.
함수가 끝나면 호출 전의 학습 모드 또는 평가 모드로 돌아간다.

In [9]:
def trace_model(model, sentence):
    """문장 하나의 학습 데이터와 다음 토큰 예측 과정을 출력한다."""
    tokens = sentence.lower().split()
    ids = [BOS_ID] + [word_to_id.get(word, UNK_ID) for word in tokens] + [EOS_ID]

    # 학습에 쓰는 x와 y를 만들고 문장 하나를 배치로 묶는다.
    training_x, training_y = LanguageModelDataset([sentence])[0]
    model_device = next(model.parameters()).device
    x = training_x.unsqueeze(0).to(model_device)

    # 생성 입력에는 정답인 <eos>와 학습용 패딩을 넣지 않는다.
    context = ids[:-1][-MAX_LEN:]
    generation_x = torch.tensor(
        context, dtype=torch.long, device=model_device
    ).unsqueeze(0)

    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            training_embedding = model.embedding(x)
            training_logits = model(x)
            generation_embedding = model.embedding(generation_x)
            logits = model(generation_x)
            last_logits = logits[0, -1]
            top_logits, top_ids = torch.topk(last_logits, k=5)
            probabilities = F.softmax(last_logits, dim=-1)
    finally:
        model.train(was_training)

    print("Original sentence:", sentence)
    print("Tokens (with <bos>/<eos>):", [id_to_word[i] for i in ids])
    print("Token IDs:", ids)
    print("Training input x:", training_x.tolist())
    print("Training input words:", [id_to_word[i] for i in training_x.tolist()])
    print("Training target y:", training_y.tolist())
    print("Training target words:", [id_to_word[i] for i in training_y.tolist()])
    print()
    print("Input shape (training):", tuple(x.shape))
    print("Embedding shape (training):", tuple(training_embedding.shape))
    print("Model logits shape (training):", tuple(training_logits.shape))
    print()
    print("Generation input IDs:", context)
    print("Generation input words:", [id_to_word[i] for i in context])
    print("Input shape (generation):", tuple(generation_x.shape))
    print("Embedding shape (generation):", tuple(generation_embedding.shape))
    print("Model logits shape (generation):", tuple(logits.shape))
    print("Last-position logits shape:", tuple(last_logits.shape))
    print("Top-5 predicted words (next token after the sentence):")
    for rank, (token_id, logit) in enumerate(zip(top_ids.tolist(), top_logits.tolist()), 1):
        word = id_to_word[token_id]
        probability = probabilities[token_id].item()
        print(f"{rank}. {word:<12} logit={logit:9.4f}  probability={probability:.6f}")

In [10]:
trace_model(rnn_model, "i want chinese food")

Original sentence: i want chinese food
Tokens (with <bos>/<eos>): ['<bos>', 'i', 'want', 'chinese', 'food', '<eos>']
Token IDs: [1, 15, 29, 8, 13, 2]
Training input x: [1, 15, 29, 8, 13, 0, 0, 0, 0, 0, 0, 0]
Training input words: ['<bos>', 'i', 'want', 'chinese', 'food', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
Training target y: [15, 29, 8, 13, 2, 0, 0, 0, 0, 0, 0, 0]
Training target words: ['i', 'want', 'chinese', 'food', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']

Input shape (training): (1, 12)
Embedding shape (training): (1, 12, 64)
Model logits shape (training): (1, 12, 33)

Generation input IDs: [1, 15, 29, 8, 13]
Generation input words: ['<bos>', 'i', 'want', 'chinese', 'food']
Input shape (generation): (1, 5)
Embedding shape (generation): (1, 5, 64)
Model logits shape (generation): (1, 5, 33)
Last-position logits shape: (33,)
Top-5 predicted words (next token after the sentence):
1. <eos>        logit=  12.5967  probability=0.

## 5. 실행 결과

`<bos> i want chinese food <eos>`의 ID는 `[1, 15, 29, 8, 13, 2]`다.
x에는 `<bos>`부터 `food`까지 들어가고, y에는 `i`부터 `<eos>`까지 들어간다.
따라서 `food` 위치의 정답은 문장 끝을 나타내는 `<eos>`다.

| 텐서 | 학습 입력을 사용할 때 | 다음 토큰을 예측할 때 |
|---|---|---|
| 입력 | `(1, 12)` | `(1, 5)` |
| 임베딩 | `(1, 12, 64)` | `(1, 5, 64)` |
| 모델 로짓 | `(1, 12, 33)` | `(1, 5, 33)` |

첫 번째 차원의 1은 문장 수다. 학습 입력의 길이는 패딩을 포함해 12이고,
다음 토큰을 예측할 때의 입력 길이는 `<bos>`와 단어 4개를 합한 5다.
64는 임베딩 차원이고, 33은 특수 토큰을 포함한 어휘 수다.

마지막 위치의 로짓 크기는 `(33,)`이다.
여기에는 `food` 다음에 올 후보 토큰 33개의 점수가 들어 있다.
패딩된 입력의 마지막 위치는 `<pad>`이므로, 다음 토큰을 예측할 때는 패딩을 넣지 않는다.

Top-5는 `food` 다음에 올 토큰의 후보 5개다.
실행 결과에서는 `<eos>`의 확률이 약 99.98%로 가장 높았다.
모델은 `food` 다음에 문장이 끝날 것으로 예측했다.